# 🏪 편의점 매출 예측 AI — 전체 실습
### 통계에 의한 AI 활용 예측 및 분석 솔루션 개발
**AI 앱크리에이터 과정 | 강사: 제조혁신 길라잡이 김사부**

---
### 📋 목차
| 단계 | 내용 | 소요시간 |
|------|------|----------|
| STEP 0 | 환경 설정 & 패키지 설치 | 5분 |
| STEP 1 | 데이터 로드 & EDA | 15분 |
| STEP 2 | 피처 엔지니어링 & 모델 학습 | 20분 |
| STEP 3 | 모델 평가 & 시각화 | 10분 |
| STEP 4 | Streamlit 앱 코드 확인 | 10분 |
| 도전과제 | 나만의 예측 모델 개선 | 자유 |


## ⚙️ STEP 0 — 환경 설정 & 패키지 설치
> Google Colab 또는 antigravity 환경에서는 아래 셀을 실행하세요.


In [ ]:
# Google Colab / antigravity 환경에서만 실행 (로컬은 skip)
# !pip install pandas numpy scikit-learn joblib matplotlib seaborn streamlit

# 한글 폰트 설치 (Linux/Colab)
# !apt-get install -y fonts-nanum -qq
# import matplotlib.font_manager as fm; fm._load_fontmanager(try_read_cache=False)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
import os, platform
warnings.filterwarnings('ignore')

# 한글 폰트 설정
system = platform.system()
if system == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif system == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print('✅ 패키지 로드 완료')
print(f'   pandas {pd.__version__}')
print(f'   numpy  {np.__version__}')
import sklearn; print(f'   sklearn {sklearn.__version__}')


## 📊 STEP 1 — 데이터 로드 & 탐색적 데이터 분석 (EDA)
**목표**: 데이터 구조 파악 → 분포 확인 → 상관관계 분석


In [ ]:
# 데이터 로드
# 같은 폴더의 data/sample_sales.csv 사용
# Google Colab이라면 파일 업로드 후 경로 수정

CSV_PATH = 'data/sample_sales.csv'  # ← 경로 수정 필요 시 여기를 바꾸세요

df = pd.read_csv(CSV_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f'Shape: {df.shape}')
df.head()


In [ ]:
# 기술 통계
df.describe().round(2)


In [ ]:
# 결측치 확인
print('결측치 현황:')
print(df.isnull().sum())

# 결측치 처리 (평균 대체)
df['temp']       = df['temp'].fillna(df['temp'].mean())
df['prev_sales'] = df['prev_sales'].fillna(df['prev_sales'].mean())
print('\n처리 후 결측치:', df.isnull().sum().sum(), '개')


In [ ]:
# 분포 시각화 (4종)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('편의점 매출 — 주요 변수 분포', fontsize=15, fontweight='bold')

# (1) 매출 히스토그램
axes[0,0].hist(df['sales'], bins=30, color='#1B4F72', edgecolor='white', alpha=0.85)
axes[0,0].axvline(df['sales'].mean(), color='#F47920', linestyle='--', linewidth=2,
                   label=f"평균: {df['sales'].mean():.1f}만원")
axes[0,0].set_title('일별 매출 분포'); axes[0,0].legend()

# (2) 요일별 평균
dow_labels = ['월','화','수','목','금','토','일']
dow_avg    = df.groupby('dayofweek')['sales'].mean()
colors     = ['#2E86C1']*5 + ['#F47920']*2
axes[0,1].bar(dow_labels, dow_avg.values, color=colors)
axes[0,1].set_title('요일별 평균 매출')

# (3) 기온 vs 매출
axes[1,0].scatter(df['temp'], df['sales'], c=df['is_weekend'],
                  cmap='RdYlGn', alpha=0.5, s=22)
axes[1,0].set_xlabel('기온(℃)'); axes[1,0].set_ylabel('매출(만원)')
axes[1,0].set_title('기온 vs 매출 (색=주말)')

# (4) 월별 추이
df['month'] = df['date'].dt.month
monthly     = df.groupby('month')['sales'].mean()
axes[1,1].plot(monthly.index, monthly.values, 'o-', color='#117A65', linewidth=2.5)
axes[1,1].set_title('월별 평균 매출 추이')

plt.tight_layout(); plt.show()


In [ ]:
# 상관관계 분석
numeric_cols = ['temp','is_weekend','holiday','event','prev_sales','month','sales']
corr = df[numeric_cols].corr()

print('sales와의 상관계수 (내림차순):')
print(corr['sales'].drop('sales').sort_values(ascending=False).round(4))

# 히트맵
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(numeric_cols))); ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('상관관계 히트맵', fontweight='bold')
plt.tight_layout(); plt.show()


## 🤖 STEP 2 — 피처 엔지니어링 & 모델 학습
**목표**: 피처 추가 → 5가지 알고리즘 비교 → 최고 성능 모델 저장


In [ ]:
# 피처 엔지니어링
df['weather_sunny'] = (df['weather'] == '맑음').astype(int)
df['weather_rain']  = (df['weather'] == '비').astype(int)
df['weather_snow']  = (df['weather'] == '눈').astype(int)

FEATURES = [
    'temp', 'is_weekend', 'holiday', 'event', 'prev_sales',
    'month', 'weather_sunny', 'weather_rain', 'weather_snow'
]
TARGET = 'sales'

X = df[FEATURES]
y = df[TARGET]

print(f'피처 수: {len(FEATURES)}개')
print(f'데이터: {len(X)}행')
print(f'매출 범위: {y.min():.1f} ~ {y.max():.1f}만원')


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics         import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'훈련: {len(X_train)}행  |  테스트: {len(X_test)}행')


In [ ]:
# 5가지 알고리즘 비교
models = {
    '선형회귀':         LinearRegression(),
    'Ridge':            Ridge(alpha=1.0),
    '의사결정나무':     DecisionTreeRegressor(max_depth=6, random_state=42),
    '랜덤포레스트':     RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
print(f\"{'모델':<20} {'RMSE':>8} {'R²':>8}\")
print('-' * 40)
for name, model in models.items():
    model.fit(X_train, y_train)
    pred  = model.predict(X_test)
    rmse  = np.sqrt(mean_squared_error(y_test, pred))
    r2    = r2_score(y_test, pred)
    results[name] = {'model': model, 'rmse': rmse, 'r2': r2}
    print(f'{name:<20} {rmse:>8.2f} {r2:>8.4f}')

best_name = min(results, key=lambda k: results[k]['rmse'])
print(f'\n★ 최고 성능: {best_name}  (RMSE: {results[best_name]["rmse"]:.2f}만원)')


In [ ]:
# 모델 저장
import joblib
best_model = results[best_name]['model']

joblib.dump(best_model, 'sales_model.pkl')
joblib.dump(FEATURES,   'features.pkl')
print('모델 저장 완료: sales_model.pkl, features.pkl')


## 📈 STEP 3 — 모델 평가 & 시각화
**목표**: 예측 vs 실제 비교 / Feature Importance 분석


In [ ]:
model    = joblib.load('sales_model.pkl')
FEATURES = joblib.load('features.pkl')

y_pred = model.predict(X_test)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)
errors = y_pred - y_test.values

print(f'RMSE : {rmse:.2f}만원')
print(f'R²   : {r2:.4f}')
print(f'목표(RMSE<15만원): {"✅ 달성!" if rmse<15 else "❌ 미달성"}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f'모델 평가 (RMSE:{rmse:.2f}  R²:{r2:.3f})', fontsize=14, fontweight='bold')

# (1) 예측 vs 실제
perfect = np.linspace(y_test.min(), y_test.max(), 100)
axes[0,0].scatter(y_test, y_pred, alpha=0.45, s=25, c=np.abs(errors), cmap='RdYlGn_r')
axes[0,0].plot(perfect, perfect, 'r--', linewidth=1.5)
axes[0,0].set_title('예측 vs 실제'); axes[0,0].set_xlabel('실제'); axes[0,0].set_ylabel('예측')

# (2) 잔차 분포
axes[0,1].hist(errors, bins=25, color='#2E86C1', edgecolor='white', alpha=0.8)
axes[0,1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_title('잔차 분포'); axes[0,1].set_xlabel('오차 (만원)')

# (3) Feature Importance
feat_imp = pd.Series(getattr(model, 'feature_importances_', getattr(model, 'coef_', [])), index=FEATURES).sort_values()
axes[1,0].barh(feat_imp.index, feat_imp.values, color='#28B463')
axes[1,0].set_title('피처 중요도'); axes[1,0].set_xlabel('중요도')

# (4) 예측 추이
sample = pd.DataFrame({'actual': y_test.values, 'pred': y_pred}).head(60)
axes[1,1].plot(sample['actual'].values, 'b-', alpha=0.7, label='실제')
axes[1,1].plot(sample['pred'].values,   'r--', alpha=0.7, label='예측')
axes[1,1].set_title('예측 추이'); axes[1,1].legend()

plt.tight_layout(); plt.show()


In [ ]:
# Feature Importance 상세 분석
feat_series = pd.Series(getattr(model, 'feature_importances_', getattr(model, 'coef_', [])), index=FEATURES).sort_values(ascending=False)
print('피처 중요도 순위:')
for i, (f, v) in enumerate(feat_series.items()):
    bar = '█' * int(v*60)
    print(f'  {i+1}. {f:<18} {bar:<30} {v*100:.1f}%')


## 🔮 STEP 4 — 새 데이터 예측 실습
**목표**: 새로운 조건을 입력해 매출을 예측한다


In [ ]:
# 나만의 조건으로 예측해보기!
# ↓↓↓ 아래 값을 바꿔서 실험해보세요 ↓↓↓

my_input = {
    'temp':          28,    # 기온 (℃)  ← 바꿔보세요!
    'is_weekend':    1,     # 주말=1, 평일=0  ← 바꿔보세요!
    'holiday':       0,     # 공휴일=1, 아니면=0
    'event':         1,     # 행사=1, 없으면=0
    'prev_sales':    210,   # 전주 동일요일 매출 (만원)  ← 바꿔보세요!
    'month':         7,     # 월 (1~12)  ← 바꿔보세요!
    'weather_sunny': 1,     # 맑음=1
    'weather_rain':  0,
    'weather_snow':  0,
}

X_new = pd.DataFrame([my_input])[FEATURES]
pred  = model.predict(X_new)[0]

print(f'입력 조건: {my_input}')
print(f'\n예측 매출: {pred:,.1f} 만원')
print(f'전주 대비: {pred - my_input["prev_sales"]:+.1f} 만원')


In [ ]:
# 기온별 예측 시뮬레이션
temps = range(-10, 41, 5)
preds = []
for t in temps:
    x = pd.DataFrame([{
        'temp': t, 'is_weekend': 0, 'holiday': 0, 'event': 0,
        'prev_sales': 200, 'month': 6,
        'weather_sunny': 1 if t > 10 else 0,
        'weather_rain': 0, 'weather_snow': 1 if t < 0 else 0,
    }])[FEATURES]
    preds.append(model.predict(x)[0])

plt.figure(figsize=(10, 5))
plt.plot(temps, preds, 'o-', color='#1B4F72', linewidth=2.5, markersize=8)
plt.fill_between(temps, preds, min(preds)*0.95, alpha=0.15, color='#1B4F72')
plt.xlabel('기온 (℃)'); plt.ylabel('예측 매출 (만원)')
plt.title('기온에 따른 예측 매출 변화 (평일, 행사없음, 6월)', fontweight='bold')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f'기온 -10℃ → {preds[0]:.1f}만원')
print(f'기온  25℃ → {preds[list(temps).index(25)]:.1f}만원')
print(f'기온  40℃ → {preds[-1]:.1f}만원')


## 🏆 도전 과제 — 나만의 모델 개선

아래 중 하나 이상을 시도해보세요:

### 🥉 브론즈 (쉬움)
- [ ] 기온 범위를 -20 ~ 45℃로 확장해서 예측 시뮬레이션
- [ ] 주말 + 공휴일 + 행사가 겹치는 날 매출 예측
- [ ] 12월(연말) 조건으로 매출 예측

### 🥈 실버 (중간)
- [ ] `dayofweek` 변수를 원핫 인코딩해서 피처로 추가
- [ ] `quarter` (분기) 변수 추가 후 성능 비교
- [ ] GridSearchCV로 최적 파라미터 탐색

### 🥇 골드 (어려움)
- [ ] 기온 × 주말 **교호작용** 변수 추가 (`temp_weekend = temp * is_weekend`)
- [ ] GradientBoosting 모델로 교체 후 RMSE 10 미만 달성
- [ ] 모델 학습 함수(`train_model()`)를 만들어 재사용 가능하게 리팩토링

### 💎 다이아 (심화)
- [ ] 실제 기상청 API 연동해서 실시간 기온으로 예측
- [ ] Streamlit 앱에 그래프 탭 추가 (월별 예측 추이, 조건별 시뮬레이션)
- [ ] 여러분 회사 데이터로 모델 학습 & 배포


In [ ]:
# 도전 과제 풀이 영역
# 여기에 코드를 작성하세요!

# ── 힌트: 교호작용 변수 추가 ──────────────────────
# df['temp_weekend'] = df['temp'] * df['is_weekend']
# df['temp_event']   = df['temp'] * df['event']

# ── 힌트: 요일 원핫 인코딩 ───────────────────────
# dow_dummies = pd.get_dummies(df['dayofweek'], prefix='dow')
# df = pd.concat([df, dow_dummies], axis=1)

# ── 힌트: GridSearchCV ───────────────────────────
# from sklearn.model_selection import GridSearchCV
# param_grid = {'n_estimators': [100, 200], 'max_depth': [None, 8, 15]}
# gs = GridSearchCV(RandomForestRegressor(random_state=42),
#                   param_grid, scoring='neg_root_mean_squared_error', cv=5)
# gs.fit(X_train, y_train)
# print('최적 파라미터:', gs.best_params_)

print('도전 과제 코드를 여기에 작성하세요!')
